In [0]:
spark.conf.set("spark.sql.shuffle.partitions", 50)

In [0]:
df = spark.read.format("csv").option("header", "true").load("/Volumes/workspace/test-schema/test-volume/Employee_Attrition.csv")
display(df)

In [0]:
history_df = spark.sql("DESCRIBE HISTORY workspace.default.high_risk_employees")
display(history_df.select("version", "timestamp", "operation", "operationParameters", "userMetadata", "isBlindAppend"))

In [0]:
table_name = "workspace.default.high_risk_employees"

# Build one dummy record that matches the table's actual schema
fields = spark.table(table_name).schema.fields

num_rows = 3

def dummy_value(field, i):
    dt = field.dataType.simpleString()
    if dt in ("int", "bigint", "smallint", "tinyint", "float", "double") or dt.startswith("decimal"):
        return str(900 + i)
    if dt == "boolean":
        return "true" if i % 2 == 0 else "false"
    if dt == "date":
        return f"date_add(current_date(), {i})"
    if "timestamp" in dt:
        return f"current_timestamp() + INTERVAL {i} MINUTES"
    return f"'DUMMY{i}'"

col_list = ", ".join(f"`{f.name}`" for f in fields)
rows = [", ".join(dummy_value(f, i) for f in fields) for i in range(num_rows)]
val_list = ", ".join(f"({row})" for row in rows)

spark.sql(f"INSERT INTO {table_name} ({col_list}) VALUES {val_list}")
print(f"Inserted {num_rows} dummy records into {table_name}")

In [0]:
delta_df = spark.read.format("delta").table("workspace.default.high_risk_employees")
display(delta_df)

In [0]:
table_name = "workspace.default.high_risk_employees"

# Read the Delta table as of a specific version (time travel)
version = 0

delta_df_version = (
    spark.read.format("delta")
    .option("versionAsOf", version)
    .table(table_name)
)
display(delta_df_version)
print(f"Showing {table_name} as of version {version}")